# Background
This notebook is for initial exploration of the US Treasury API endpoints.

# Imports

In [ ]:
# import sys
# import argparse
import json
from urllib import request
from urllib.parse import quote, urlencode

In [ ]:
import pandas as pd

In [ ]:
baseurl = "https://api.fiscaldata.treasury.gov/services/api/fiscal_service"

# Debt To The Penny

In [ ]:
endpoint = "v2/accounting/od/debt_to_penny"

In [ ]:
params = {
    "format": "csv",
    "filter": "record_date:gte:2026-06-30"
}

In [ ]:
qstr = urlencode(params)

In [ ]:
fullurl = f"{baseurl}/{endpoint}?{qstr}"
print(fullurl)

In [ ]:
df = None

In [ ]:
with request.urlopen(fullurl) as conn: 
    df = pd.read_csv(conn)

In [ ]:
df

In [ ]:
df.describe()

# Revenue Collections (RCM)

In [ ]:
endpoint = "v2/revenue/rcm"
params = {
    "format": "csv",
    "filter": "record_date:gte:2026-01-01",
    "page[size]": 3000
}
qstr = urlencode(params)
fullurl = f"{baseurl}/{endpoint}?{qstr}"
print(fullurl)

In [ ]:
df = None

In [ ]:
with request.urlopen(fullurl) as conn: 
    df = pd.read_csv(conn)

In [ ]:
if df.net_collections_amt.str.contains("'").any():
    df = df.assign(
        net_collections_amt = lambda df0: df0.net_collections_amt.str.lstrip("'").astype(float)
    )

In [ ]:
df.net_collections_amt.astype(float)

In [ ]:
df

In [ ]:
df.groupby(["record_calendar_year", "record_calendar_month", "tax_category_desc"]).agg(
    net_collections_amt=("net_collections_amt","sum")
).assign(
    net_collections_amt_bn = lambda df0: df0.net_collections_amt / 1.0e9
)

In [ ]:
df.groupby(["tax_category_desc", "electronic_category_desc", "channel_type_desc"]).agg(
    net_collections_amt=("net_collections_amt","sum")
).assign(
    net_collections_amt_bn = lambda df0: df0.net_collections_amt / 1.0e9
)